# 08 — Reusable Pipeline Function Pattern

Wrap full transformation into reusable function.

Process:

INPUT DF
  |
  v
PIPELINE FUNCTION
  |
  +--> silver
  +--> quarantine

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("reusable-pipeline")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [
        ("e1","u1",10.0),
        ("e2",None,20.0),
        ("e3","u2",-5.0),
    ],
    ["event_id","user_id","amount"]
)

df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|   NULL|  20.0|
|      e3|     u2|  -5.0|
+--------+-------+------+



## Step 2 — Pipeline function

In [3]:
def build_pipeline(df):
    prepared = (
        df
        .withColumn("error_reason",
            F.when(F.col("user_id").isNull(), F.lit("missing_user"))
             .when(F.col("amount") < 0, F.lit("negative_amount"))
        )
        .withColumn("is_valid", F.col("error_reason").isNull())
    )

    return {
        "silver": prepared.filter("is_valid"),
        "quarantine": prepared.filter("NOT is_valid")
    }

## Step 3 — Run

In [4]:
result = build_pipeline(df)

result["silver"].show()
result["quarantine"].show()

+--------+-------+------+------------+--------+
|event_id|user_id|amount|error_reason|is_valid|
+--------+-------+------+------------+--------+
|      e1|     u1|  10.0|        NULL|    true|
+--------+-------+------+------------+--------+

+--------+-------+------+---------------+--------+
|event_id|user_id|amount|   error_reason|is_valid|
+--------+-------+------+---------------+--------+
|      e2|   NULL|  20.0|   missing_user|   false|
|      e3|     u2|  -5.0|negative_amount|   false|
+--------+-------+------+---------------+--------+



## Step 4 — Control

In [5]:
print("input:", df.count())
print("silver:", result["silver"].count())
print("quarantine:", result["quarantine"].count())

input: 3
silver: 1
quarantine: 2


In [6]:
spark.stop()